# LinkRAG — serve an open model from a free Colab GPU

Serves **Qwen2.5-7B-Instruct** through Ollama's OpenAI-compatible endpoint and exposes it with a
Cloudflare quick tunnel (no account). LinkRAG then uses it as backend `colab`, for the answerer
and, once its agreement is measured, the judge. Cost: $0.

1. **Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom.
2. The last setup cell prints one line. Run it **on your machine**, in the shell where LinkRAG runs:
   `export LINKRAG_COLAB_BASE_URL=https://<random>.trycloudflare.com/v1`
3. Select the backend there: `LINKRAG_LLM_BACKEND=colab` (answerer) and/or `LINKRAG_JUDGE_BACKEND=colab` (judge).
   Measure the judge first: `python scripts/eval/judge_agreement.py --judge colab --max-cost 0`.

**Free sessions disconnect** after a few hours, or sooner when idle. Every LinkRAG run caches each
reply on disk, so after a disconnect you re-run this notebook, export the new URL, and re-run the
same command: finished calls replay from the cache and only the rest are sent.

**Exposure.** Ollama has no authentication, so anyone who has the tunnel URL can use this GPU.
Nothing of yours is on it apart from the prompts you send. The URL is random; stop the tunnel
(last cell) or end the runtime when you are done.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

## 2. Install and start Ollama

The modality-only answerer sends up to ~30k tokens and Ollama's default context is 4k, so the
context is raised to 32k. That fits the 7B model on a T4.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
!OLLAMA_HOST=0.0.0.0:11434 OLLAMA_CONTEXT_LENGTH=32768 nohup ollama serve > ollama.log 2>&1 &
!sleep 5 && curl -s http://127.0.0.1:11434/api/version

## 3. Pull the model

`qwen2.5:7b-instruct` (about 4.7 GB) is the default in `configs/default.yaml` (`models.llm_backends.colab.model`).

**Optional 14B (quantised).** `qwen2.5:14b-instruct-q4_K_M` is about 9 GB of weights. At 32k context,
its KV cache adds roughly 6 GB more (an estimate), which exceeds a T4's 15 GB. So restart Ollama with
`OLLAMA_CONTEXT_LENGTH=16384` for it. Long modality-only prompts then get truncated, so only use it
for the judge (short prompts). Set `model: qwen2.5:14b-instruct-q4_K_M` in the `colab` block if you
switch. To check it fits: after the first request, `ollama ps` should say **100% GPU**. Any CPU
share means it spilled and will be slow. `nvidia-smi` shows the memory in use.

In [ ]:
!ollama pull qwen2.5:7b-instruct
# !ollama pull qwen2.5:14b-instruct-q4_K_M     # optional; see the note above
!ollama list

In [ ]:
# one warm-up request, then check where the model is loaded
!curl -s http://127.0.0.1:11434/v1/chat/completions -H 'Content-Type: application/json' \
  -d '{"model": "qwen2.5:7b-instruct", "messages": [{"role": "user", "content": "Reply with OK."}], "temperature": 0}' | head -c 300
!ollama ps
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

## 4. Open the tunnel and print the export line

`--http-host-header` makes the forwarded requests look local to Ollama, so its host check passes.

In [ ]:
import re, subprocess, time
subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
               "-O cloudflared && chmod +x cloudflared", shell=True, check=True)
tunnel = subprocess.Popen("./cloudflared tunnel --no-autoupdate --url http://127.0.0.1:11434 "
                          "--http-host-header localhost:11434 > tunnel.log 2>&1", shell=True)
url = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        url = m.group(0)
        break
assert url, "no tunnel URL yet: see tunnel.log"
print("Run this on your machine (the shell where LinkRAG runs):\n")
print(f"export LINKRAG_COLAB_BASE_URL={url}/v1")

In [ ]:
# the tunnel answers from outside (DNS for a new quick tunnel can take ~30 s)
time.sleep(20)
!curl -s {url}/v1/models | head -c 300

## 5. Keep-alive

Keeps the session busy while LinkRAG runs, and prints whether the server still answers. It does not
stop Colab's time limits, only the idle disconnect. Stop it with the ■ button; the server and tunnel
keep running.

In [ ]:
import urllib.request
while True:
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=10).read()
        print(time.strftime("%H:%M:%S"), "ollama up ·", url)
    except Exception as exc:
        print(time.strftime("%H:%M:%S"), "ollama NOT answering:", exc)
    time.sleep(60)

## 6. Stop

In [ ]:
tunnel.terminate()
!pkill -f "ollama serve"